# Module 4: Pandas for Machine Learning

**Topics Covered:**
- Series & DataFrame creation
- Loading & inspecting data
- Selection, filtering, sorting
- Handling missing values
- Feature engineering (apply, map, groupby)
- Merging & reshaping
- Encoding categorical variables
- Train/val/test splitting

> Pandas is your primary tool for EDA and feature engineering — the most time-consuming part of real ML projects.

In [ ]:
import pandas as pd
import numpy as np
print(f"Pandas: {pd.__version__}, NumPy: {np.__version__}")

---
## 1. Creating DataFrames

In [ ]:
# From dict — most common in practice
np.random.seed(42)

df = pd.DataFrame({
    'age':        np.random.randint(18, 70, 10),
    'income':     np.random.randint(30000, 120000, 10),
    'education':  np.random.choice(['high_school', 'bachelor', 'master', 'phd'], 10),
    'credit_score': np.random.randint(550, 850, 10),
    'default':    np.random.choice([0, 1], 10, p=[0.7, 0.3])
})

df

In [ ]:
# Inspect the DataFrame
print("=== Shape ===")
print(df.shape)

print("\n=== dtypes ===")
print(df.dtypes)

print("\n=== Basic Stats ===")
print(df.describe().round(2))

print("\n=== Missing Values ===")
print(df.isnull().sum())

---
## 2. Selection & Filtering

In [ ]:
# Column selection
ages = df['age']                          # Series
features = df[['age', 'income', 'credit_score']]  # DataFrame

# Row selection
print("Row 0:", df.iloc[0].to_dict())     # by position
print("Rows 2-4:")
print(df.iloc[2:5])

# Boolean filtering
high_income = df[df['income'] > 80000]
at_risk = df[(df['credit_score'] < 650) & (df['default'] == 1)]

print(f"\nHigh income count: {len(high_income)}")
print(f"At-risk count: {len(at_risk)}")
print(f"At-risk records:\n{at_risk}")

In [ ]:
# query() — readable filtering
result = df.query("age > 30 and income > 60000 and default == 0")
print(f"Low-risk adults (age>30, income>60k, no default):")
print(result)

### Exercise 4.1 — Selection & Filtering
Using the `df` created above:
1. Select only `education` and `default` columns for rows where `credit_score >= 700`
2. Find the average `income` grouped by `default` (0 vs 1)
3. Find the row with the maximum `credit_score` using `.idxmax()`

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION
# 1
good_credit = df.loc[df['credit_score'] >= 700, ['education', 'default']]
print("1. Good credit - education & default:")
print(good_credit)

# 2
avg_income_by_default = df.groupby('default')['income'].mean()
print(f"\n2. Avg income by default status:\n{avg_income_by_default}")

# 3
best_credit_idx = df['credit_score'].idxmax()
print(f"\n3. Best credit score row (index {best_credit_idx}):")
print(df.loc[best_credit_idx])

---
## 3. Handling Missing Values

In [ ]:
# Create a dataset with realistic missing patterns
np.random.seed(7)
n = 20
df_messy = pd.DataFrame({
    'age':          np.where(np.random.rand(n) < 0.1, np.nan, np.random.randint(18, 70, n).astype(float)),
    'income':       np.where(np.random.rand(n) < 0.2, np.nan, np.random.randint(30000, 120000, n).astype(float)),
    'bmi':          np.where(np.random.rand(n) < 0.15, np.nan, np.random.uniform(18, 40, n)),
    'smoke':        np.where(np.random.rand(n) < 0.3, np.nan, np.random.choice(['yes','no'], n)),
    'target':       np.random.choice([0, 1], n)
})

print("Missing value counts:")
print(df_messy.isnull().sum())
print(f"\nMissing %:")
print((df_messy.isnull().mean() * 100).round(1))

In [ ]:
# Strategies for handling missing values
df_filled = df_messy.copy()

# Numeric: fill with median (robust to outliers)
df_filled['age'].fillna(df_filled['age'].median(), inplace=True)
df_filled['income'].fillna(df_filled['income'].median(), inplace=True)
df_filled['bmi'].fillna(df_filled['bmi'].mean(), inplace=True)

# Categorical: fill with mode
df_filled['smoke'].fillna(df_filled['smoke'].mode()[0], inplace=True)

print("After filling:")
print(df_filled.isnull().sum())
print(f"\nFilled age median: {df_messy['age'].median():.1f}")

---
## 4. Feature Engineering

In [ ]:
# Building the feature engineering pipeline step by step
np.random.seed(42)
df_fe = pd.DataFrame({
    'age':        np.random.randint(18, 70, 200),
    'income':     np.random.randint(20000, 150000, 200),
    'debt':       np.random.randint(0, 80000, 200),
    'education':  np.random.choice(['high_school','bachelor','master','phd'], 200),
    'employed':   np.random.choice([True, False], 200)
})

# 1. Derived features
df_fe['debt_to_income'] = df_fe['debt'] / df_fe['income'].replace(0, np.nan)
df_fe['income_per_age'] = df_fe['income'] / df_fe['age']

# 2. Binning (age groups)
df_fe['age_group'] = pd.cut(
    df_fe['age'],
    bins=[0, 25, 35, 50, 100],
    labels=['young', 'adult', 'middle_aged', 'senior']
)

# 3. Apply custom function
edu_rank = {'high_school': 1, 'bachelor': 2, 'master': 3, 'phd': 4}
df_fe['edu_rank'] = df_fe['education'].map(edu_rank)

# 4. Boolean to int
df_fe['employed_int'] = df_fe['employed'].astype(int)

print(df_fe[['age', 'income', 'debt', 'debt_to_income', 'age_group', 'edu_rank']].head(8))

In [ ]:
# Encoding categoricals — crucial before ML

# 1. One-Hot Encoding
df_ohe = pd.get_dummies(df_fe, columns=['education', 'age_group'], drop_first=True)
print(f"Shape after OHE: {df_ohe.shape}")
print(f"New columns: {[c for c in df_ohe.columns if 'education' in c or 'age_group' in c]}")

# 2. Ordinal Encoding (already done with map above)
print(f"\nEdu rank distribution:\n{df_fe['edu_rank'].value_counts().sort_index()}")

### Exercise 4.2 — Feature Engineering
Create a customer churn feature engineering pipeline:
```python
np.random.seed(42)
customers = pd.DataFrame({
    'customer_id': range(1, 201),
    'tenure_months': np.random.randint(1, 72, 200),
    'monthly_charges': np.random.uniform(20, 120, 200),
    'total_charges': None,  # to be computed
    'num_services': np.random.randint(1, 8, 200),
    'contract': np.random.choice(['month-to-month', 'one-year', 'two-year'], 200),
    'churned': np.random.choice([0, 1], 200, p=[0.73, 0.27])
})
```
1. Compute `total_charges = tenure_months * monthly_charges`
2. Create `avg_monthly = total_charges / tenure_months`
3. Create `high_value` boolean: True if `monthly_charges > 75`
4. One-hot encode `contract` (drop_first=True)
5. Compute churn rate per contract type

In [ ]:
np.random.seed(42)
customers = pd.DataFrame({
    'customer_id': range(1, 201),
    'tenure_months': np.random.randint(1, 72, 200),
    'monthly_charges': np.random.uniform(20, 120, 200).round(2),
    'num_services': np.random.randint(1, 8, 200),
    'contract': np.random.choice(['month-to-month', 'one-year', 'two-year'], 200),
    'churned': np.random.choice([0, 1], 200, p=[0.73, 0.27])
})
# YOUR CODE HERE


In [ ]:
# SOLUTION
np.random.seed(42)
customers = pd.DataFrame({
    'customer_id': range(1, 201),
    'tenure_months': np.random.randint(1, 72, 200),
    'monthly_charges': np.random.uniform(20, 120, 200).round(2),
    'num_services': np.random.randint(1, 8, 200),
    'contract': np.random.choice(['month-to-month', 'one-year', 'two-year'], 200),
    'churned': np.random.choice([0, 1], 200, p=[0.73, 0.27])
})

# 1 & 2
customers['total_charges'] = customers['tenure_months'] * customers['monthly_charges']
customers['avg_monthly']   = customers['total_charges'] / customers['tenure_months']

# 3
customers['high_value'] = customers['monthly_charges'] > 75

# 4
customers_enc = pd.get_dummies(customers, columns=['contract'], drop_first=True)
print(f"Shape after encoding: {customers_enc.shape}")
print(f"Contract columns: {[c for c in customers_enc.columns if 'contract' in c]}")

# 5
churn_by_contract = customers.groupby('contract')['churned'].mean().round(3)
print(f"\nChurn rate by contract:\n{churn_by_contract}")

---
## 5. GroupBy & Aggregation

In [ ]:
# GroupBy — essential for EDA
summary = df_fe.groupby('education').agg(
    count=('age', 'count'),
    avg_age=('age', 'mean'),
    avg_income=('income', 'mean'),
    avg_debt_ratio=('debt_to_income', 'mean')
).round(2)

print("Education Group Summary:")
print(summary.sort_values('avg_income', ascending=False))

---
## 6. Train/Val/Test Split with Pandas

In [ ]:
from sklearn.model_selection import train_test_split

# Create a clean ML-ready dataset
np.random.seed(42)
n = 500
ml_df = pd.DataFrame({
    'feature_1': np.random.randn(n),
    'feature_2': np.random.randn(n),
    'feature_3': np.random.uniform(0, 10, n),
    'category':  np.random.choice(['A', 'B', 'C'], n),
    'target':    np.random.choice([0, 1], n)
})

# Encode and split
ml_df_enc = pd.get_dummies(ml_df, columns=['category'], drop_first=True)

feature_cols = [c for c in ml_df_enc.columns if c != 'target']
X = ml_df_enc[feature_cols]
y = ml_df_enc['target']

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Class balance — Train: {y_train.mean():.2f}, Val: {y_val.mean():.2f}, Test: {y_test.mean():.2f}")

---
## Challenge: Full EDA & Feature Engineering Pipeline

Build a complete preprocessing pipeline for the Titanic-style dataset below:

1. **Inspect**: shape, dtypes, missing values, class balance
2. **Clean**: fill missing `age` with median, fill missing `embarked` with mode, drop `cabin` (too many NaN)
3. **Engineer**: create `family_size = sibsp + parch + 1`, `is_alone = (family_size == 1)`, `title` extracted from `name`
4. **Encode**: one-hot `sex` and `embarked`; ordinal encode `pclass`
5. **Analyze**: compute survival rate by `pclass`, `sex`, and `title`

In [ ]:
np.random.seed(42)
n = 200
names = [
    'Mr. John Smith', 'Mrs. Jane Doe', 'Miss. Alice Brown', 'Master. Tom Lee',
    'Dr. Bob Wilson', 'Mr. Carlos Diaz', 'Mrs. Maria Chen', 'Miss. Sara Kim'
]
titanic = pd.DataFrame({
    'survived': np.random.choice([0, 1], n, p=[0.62, 0.38]),
    'pclass':   np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
    'name':     np.random.choice(names, n),
    'sex':      np.random.choice(['male', 'female'], n, p=[0.65, 0.35]),
    'age':      np.where(np.random.rand(n) < 0.2, np.nan, np.random.uniform(1, 80, n).round(0)),
    'sibsp':    np.random.randint(0, 5, n),
    'parch':    np.random.randint(0, 4, n),
    'fare':     np.random.uniform(5, 300, n).round(2),
    'cabin':    np.where(np.random.rand(n) < 0.77, np.nan, 'C' + pd.Series(np.random.randint(10, 99, n)).astype(str)),
    'embarked': np.where(np.random.rand(n) < 0.02, np.nan, np.random.choice(['S', 'C', 'Q'], n, p=[0.72, 0.19, 0.09]))
})

print(titanic.head())
print(f"\nShape: {titanic.shape}")

In [ ]:
# YOUR CODE HERE — Steps 1-5


In [ ]:
# SOLUTION
df_t = titanic.copy()

# 1. Inspect
print("=== Missing values ===")
print(df_t.isnull().sum())
print(f"\nSurvival rate: {df_t['survived'].mean():.2%}")

# 2. Clean
df_t['age'].fillna(df_t['age'].median(), inplace=True)
df_t['embarked'].fillna(df_t['embarked'].mode()[0], inplace=True)
df_t.drop(columns=['cabin'], inplace=True)

# 3. Engineer features
df_t['family_size'] = df_t['sibsp'] + df_t['parch'] + 1
df_t['is_alone'] = (df_t['family_size'] == 1).astype(int)
df_t['title'] = df_t['name'].str.extract(r'([A-Za-z]+)\.')

# 4. Encode
df_t = pd.get_dummies(df_t, columns=['sex', 'embarked'], drop_first=True)
df_t.drop(columns=['name'], inplace=True)

print("\n=== Processed DataFrame ===")
print(df_t.head())
print(f"\nFinal shape: {df_t.shape}")

# 5. Analyze
print("\n=== Survival by Pclass ===")
print(titanic.groupby('pclass')['survived'].mean().round(3))

print("\n=== Survival by Sex ===")
print(titanic.groupby('sex')['survived'].mean().round(3))

print("\n=== Survival by Title ===")
title_col = df_t['title'] if 'title' in df_t.columns else titanic['name'].str.extract(r'([A-Za-z]+)\.')[0]
title_survival = pd.DataFrame({'title': title_col, 'survived': titanic['survived']})
print(title_survival.groupby('title')['survived'].mean().round(3).sort_values(ascending=False))